# MCD-rPPG Dataset - Correct Paths EDA

## Metadata is in db.csv, File Paths are Relative

### Key Understanding:
- **Metadata (vital signs, demographics)**: Already in db.csv columns
- **File paths**: Relative paths in db.csv columns (ecg, ppg, video, meta, ppg_sync)
- **Base path**: DATASET_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/'
- **Full path**: os.path.join(DATASET_PATH, relative_path_from_db)

### Example from db.csv:
patient_id | weight | ecg | ppg | video | meta | ppg_sync
1020 | 55.0 | ecg/1020_after.json | ppg/1020_after.PW | video/1020_FullHDwebcam_after.avi | meta/1020_FullHDwebcam_after.txt | ppg_sync/1020_FullHDwebcam_after.txt

### Full Path Construction:
```python
ecg_full = os.path.join(DATASET_PATH, row['ecg'])
ppg_full = os.path.join(DATASET_PATH, row['ppg'])
video_full = os.path.join(DATASET_PATH, row['video'])
meta_full = os.path.join(DATASET_PATH, row['meta'])
ppg_sync_full = os.path.join(DATASET_PATH, row['ppg_sync'])
```

---

### Setup and Configuration

In [ ]:
# Install required packages
!pip install imageio[ffmpeg] -q
!pip install mediapipe -q
!pip install scikit-video -q

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from pathlib import Path
import json
import imageio
import skvideo.io
import cv2
from IPython.display import display, HTML
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# CORRECT CONFIGURATION - ALL PATHS RELATIVE TO DATASET_PATH
DATASET_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/'
DB_PATH = os.path.join(DATASET_PATH, 'db.csv')

print('=' * 80)
print('CONFIGURATION - CORRECT PATHS')
print('=' * 80)
print(f'DATASET_PATH: {DATASET_PATH}')
print(f'DB_PATH: {DB_PATH}')
print()
print('File path construction:')
print('  ecg_full = os.path.join(DATASET_PATH, row["ecg"])')
print('  ppg_full = os.path.join(DATASET_PATH, row["ppg"])')
print('  video_full = os.path.join(DATASET_PATH, row["video"])')
print('  meta_full = os.path.join(DATASET_PATH, row["meta"])')
print('  ppg_sync_full = os.path.join(DATASET_PATH, row["ppg_sync"])')

## 1. Load Database and Verify Structure

In [ ]:
# Load the database
df = pd.read_csv(DB_PATH)

print('=' * 80)
print('DATABASE STRUCTURE')
print('=' * 80)
print(f'Shape: {df.shape}')
print(f'Columns: {len(df.columns)}')
print(f'Rows: {len(df)}')

print('Column Names and Types:')
for col in df.columns:
    dtype = df[col].dtype
    missing = df[col].isna().sum()
    print(f'  {col:25s} : {str(dtype):10s} : {missing:4d} missing')

print()
print('First 3 rows:')
display(df.head(3))

## 2. Verify File Paths Construction

In [ ]:
# Verify that file paths can be constructed correctly
print('=' * 80)
print('FILE PATH CONSTRUCTION VERIFICATION')
print('=' * 80)

file_columns = ['ecg', 'ppg', 'video', 'meta', 'ppg_sync']

# Test with first row
sample_row = df.iloc[0]
print(f'Sample row (patient_id={sample_row["patient_id"]}):')
print()

for col in file_columns:
    if col in df.columns:
        relative_path = sample_row[col]
        full_path = os.path.join(DATASET_PATH, relative_path) if not os.path.isabs(relative_path) else relative_path
        exists = os.path.exists(full_path)
        print(f'  {col:12s}: {relative_path}')
        print(f'           -> {full_path}')
        print(f'           Exists: {exists}')
        print()

# Check all file columns for missing values
print('File column completeness:')
for col in file_columns:
    if col in df.columns:
        total = df[col].notna().sum()
        missing = df[col].isna().sum()
        print(f'  {col:12s}: {total:4d} valid, {missing:4d} missing')

## 3. Extract Metadata from db.csv (NOT from separate files)

In [ ]:
# ALL METADATA IS IN db.csv - NO SEPARATE METADATA FILES NEEDED!
print('=' * 80)
print('METADATA EXTRACTION FROM db.csv')
print('=' * 80)
print('Metadata is ALREADY in the database columns!')
print()

# Vital signs columns (metadata)
vital_signs = ['weight', 'height', 'bmi', 'age', 'sex', 
               'upper_ap', 'lower_ap', 'saturation', 'temperature', 
               'hemoglobin', 'glycated_hemoglobin', 'cholesterol', 
               'respiratory', 'rigidity', 'pulse', 'stress']

print('Vital Signs (Metadata) Columns:')
for col in vital_signs:
    if col in df.columns:
        print(f'  OK {col}')
    else:
        print(f'  MISSING {col}')

print()

# Session information columns
session_cols = ['step', 'camera', 'view']
print('Session Information Columns:')
for col in session_cols:
    if col in df.columns:
        unique_vals = df[col].unique()[:10]
        print(f'  OK {col}: {unique_vals}')
    else:
        print(f'  MISSING {col}')

print()

# File path columns
print('File Path Columns (Relative Paths):')
for col in file_columns:
    if col in df.columns:
        print(f'  OK {col}')
    else:
        print(f'  MISSING {col}')

print()
print('=' * 80)
print('IMPORTANT: All metadata is in db.csv columns!')
print('File paths in db.csv are RELATIVE and need DATASET_PATH prefix!')
print('=' * 80)

## 4. Prepare Complete Metadata DataFrame

In [ ]:
# Create a complete metadata dataframe with full paths
print('=' * 80)
print('PREPARING COMPLETE METADATA')
print('=' * 80)

# Copy the original dataframe
meta_df = df.copy()

# Add full paths for all file columns
for col in file_columns:
    if col in meta_df.columns:
        meta_df[f'{col}_full'] = meta_df[col].apply(
            lambda x: os.path.join(DATASET_PATH, x) if not os.path.isabs(x) else x
        )

# Add extracted metadata
meta_df['subject_id'] = meta_df['patient_id']
meta_df['condition'] = meta_df['step']
meta_df['camera_type'] = meta_df['camera']
meta_df['view_type'] = meta_df['view']

print('Enhanced Metadata Columns:')
enhanced_cols = [col for col in meta_df.columns if col not in df.columns or col in ['subject_id', 'condition', 'camera_type', 'view_type']]
for col in enhanced_cols:
    print(f'  OK {col}')

print()

# Show sample with full paths
sample_cols = ['patient_id', 'condition', 'camera_type', 'view_type', 
               'ecg_full', 'ppg_full', 'video_full', 'meta_full', 'ppg_sync_full']
display(meta_df[sample_cols].head(3))

## 5. Vital Signs Analysis (Metadata from db.csv)

In [ ]:
# Analyze vital signs from db.csv
print('=' * 80)
print('VITAL SIGNS ANALYSIS (FROM db.csv)')
print('=' * 80)

# Units for each vital sign
def get_unit(col):
    units = {
        'weight': 'kg', 'height': 'cm', 'bmi': 'kg/m2', 'age': 'years',
        'upper_ap': 'mmHg', 'lower_ap': 'mmHg', 'saturation': '%',
        'temperature': 'C', 'hemoglobin': 'g/dL', 'glycated_hemoglobin': '%',
        'cholesterol': 'mmol/L', 'respiratory': 'bpm', 'rigidity': 'm/s',
        'pulse': 'bpm', 'stress': 'score'
    }
    return units.get(col, '')

# Create statistics table
vital_stats = []
for col in vital_signs:
    if col in meta_df.columns:
        meta_df[col] = pd.to_numeric(meta_df[col], errors='coerce')
        vital_stats.append({
            'Vital Sign': col.replace('_', ' ').title(),
            'Min': f'{meta_df[col].min():.2f}',
            'Max': f'{meta_df[col].max():.2f}',
            'Mean': f'{meta_df[col].mean():.2f}',
            'Std': f'{meta_df[col].std():.2f}',
            'Unit': get_unit(col),
            'Missing': meta_df[col].isna().sum()
        })

vital_df = pd.DataFrame(vital_stats)
display(vital_df)

In [ ]:
# Plot distributions
plt.figure(figsize=(18, 12))
for i, col in enumerate(vital_signs[:12], 1):
    plt.subplot(4, 3, i)
    if col in meta_df.columns:
        sns.histplot(meta_df[col].dropna(), kde=True, bins=30, color='skyblue')
        plt.title(f'{col.replace("_", " ").title()}')
        plt.xlabel('')
plt.tight_layout()
plt.show()

# Plot remaining vital signs
if len(vital_signs) > 12:
    plt.figure(figsize=(12, 4))
    for i, col in enumerate(vital_signs[12:], 1):
        plt.subplot(1, len(vital_signs[12:]), i)
        if col in meta_df.columns:
            sns.histplot(meta_df[col].dropna(), kde=True, bins=30, color='salmon')
            plt.title(f'{col.replace("_", " ").title()}')
    plt.tight_layout()
    plt.show()

## 6. Video Analysis with Correct Paths

In [ ]:
# Analyze videos using correct full paths
print('=' * 80)
print('VIDEO ANALYSIS WITH CORRECT PATHS')
print('=' * 80)

# Get sample videos with full paths
sample_videos = meta_df.dropna(subset=['video_full']).head(3)
print(f'Analyzing {len(sample_videos)} sample videos...')

for idx, row in sample_videos.iterrows():
    video_full = row['video_full']
    
    if os.path.exists(video_full):
        try:
            reader = imageio.get_reader(video_full, 'ffmpeg')
            meta_data = reader.get_meta_data()
            
            # FIXED: Use count_frames() for accurate count
            n_frames = reader.count_frames()
            fps = meta_data.get('fps', 30.0)
            duration = n_frames / fps if fps > 0 else 0
            size = meta_data.get('size', (0, 0))
            
            print(f'Video {idx + 1}: {os.path.basename(video_full)}')
            print(f'  Subject: {row["patient_id"]}, Condition: {row["condition"]}, Camera: {row["camera_type"]}, View: {row["view_type"]}')
            print(f'  Resolution: {size[0]}x{size[1]}')
            print(f'  FPS: {fps:.2f}')
            print(f'  Frames: {n_frames}')
            print(f'  Duration: {duration:.2f} seconds')
            
            first_frame = reader.get_next_data()
            reader.close()
            
            plt.figure(figsize=(8, 6))
            plt.imshow(first_frame)
            plt.title(f'{os.path.basename(video_full)} - First Frame')
            plt.axis('off')
            plt.show()
            
        except Exception as e:
            print(f'Error reading {video_full}: {e}')
    else:
        print(f'Video not found: {video_full}')

## 7. PPG Signal Analysis with Correct Paths

In [ ]:
# Analyze PPG signals using correct full paths
print('=' * 80)
print('PPG SIGNAL ANALYSIS WITH CORRECT PATHS')
print('=' * 80)

# Get sample PPG files with full paths
sample_ppgs = meta_df.dropna(subset=['ppg_full']).head(3)
print(f'Analyzing {len(sample_ppgs)} sample PPG files...')

for idx, row in sample_ppgs.iterrows():
    ppg_full = row['ppg_full']
    
    if os.path.exists(ppg_full):
        try:
            # Handle both .npy and .PW files
            if ppg_full.endswith('.npy'):
                ppg_signal = np.load(ppg_full)
            elif ppg_full.endswith('.PW'):
                ppg_signal = np.loadtxt(ppg_full)
            else:
                ppg_signal = np.load(ppg_full)
            
            print(f'PPG {idx + 1}: {os.path.basename(ppg_full)}')
            print(f'  Subject: {row["patient_id"]}, Condition: {row["condition"]}')
            print(f'  Shape: {ppg_signal.shape}')
            print(f'  Dtype: {ppg_signal.dtype}')
            print(f'  Duration: {len(ppg_signal) / 100:.2f} seconds (at 100Hz)')
            print(f'  Min: {ppg_signal.min():.2f}, Max: {ppg_signal.max():.2f}, Mean: {ppg_signal.mean():.2f}')
            
            plt.figure(figsize=(15, 4))
            plt.plot(ppg_signal[:1000], color='red', linewidth=1)
            plt.title(f'PPG Signal - {os.path.basename(ppg_full)} (First 1000 samples)')
            plt.xlabel('Sample Index')
            plt.ylabel('PPG Value')
            plt.grid(True, alpha=0.3)
            plt.show()
            
        except Exception as e:
            print(f'Error loading {ppg_full}: {e}')
    else:
        print(f'PPG file not found: {ppg_full}')

## 8. PPG_Sync Analysis with Correct Paths

In [ ]:
# Analyze PPG sync files using correct full paths
print('=' * 80)
print('PPG_SYNC ANALYSIS WITH CORRECT PATHS')
print('=' * 80)

# Get sample PPG sync files with full paths
sample_ppg_syncs = meta_df.dropna(subset=['ppg_sync_full']).head(3)
print(f'Analyzing {len(sample_ppg_syncs)} sample PPG sync files...')

for idx, row in sample_ppg_syncs.iterrows():
    ppg_sync_full = row['ppg_sync_full']
    
    if os.path.exists(ppg_sync_full):
        try:
            if ppg_sync_full.endswith('.txt'):
                df_sync = pd.read_csv(ppg_sync_full, sep='\s+', header=None)
            elif ppg_sync_full.endswith('.npy'):
                df_sync = np.load(ppg_sync_full)
                df_sync = pd.DataFrame(df_sync)
            else:
                df_sync = pd.read_csv(ppg_sync_full)
            
            sync_rows = len(df_sync)
            print(f'PPG Sync {idx + 1}: {os.path.basename(ppg_sync_full)}')
            print(f'  Rows: {sync_rows}')
            print(f'  Shape: {df_sync.shape}')
            
        except Exception as e:
            print(f'Error reading {ppg_sync_full}: {e}')
    else:
        print(f'PPG sync file not found: {ppg_sync_full}')

## 9. Video vs PPG_Sync Comparison

In [ ]:
# Compare video frames with PPG sync data
print('=' * 80)
print('VIDEO VS PPG_SYNC COMPARISON')
print('=' * 80)

# Get sample pairs
sample_df = meta_df.dropna(subset=['video_full', 'ppg_sync_full']).head(3)

for idx, row in sample_df.iterrows():
    video_full = row['video_full']
    ppg_sync_full = row['ppg_sync_full']
    
    print(f'Sample {idx + 1}: Subject={row["patient_id"]}, Condition={row["condition"]}, Camera={row["camera_type"]}')
    
    # Analyze video
    if os.path.exists(video_full):
        try:
            reader = imageio.get_reader(video_full, 'ffmpeg')
            n_frames = reader.count_frames()
            fps = reader.get_meta_data().get('fps', 30.0)
            video_duration = n_frames / fps if fps > 0 else 0
            reader.close()
            print(f'  Video: {n_frames} frames, {fps:.2f} FPS, {video_duration:.2f}s')
        except Exception as e:
            print(f'  Error reading video: {e}')
    else:
        print(f'  Video not found: {video_full}')
    
    # Analyze PPG sync
    if os.path.exists(ppg_sync_full):
        try:
            if ppg_sync_full.endswith('.txt'):
                df_sync = pd.read_csv(ppg_sync_full, sep='\s+', header=None)
            elif ppg_sync_full.endswith('.npy'):
                df_sync = np.load(ppg_sync_full)
                df_sync = pd.DataFrame(df_sync) if df_sync.ndim > 1 else pd.DataFrame(df_sync.reshape(-1, 1))
            else:
                df_sync = pd.read_csv(ppg_sync_full)
            
            sync_rows = len(df_sync)
            print(f'  PPG Sync: {sync_rows} rows')
            
            # Compare
            if n_frames > 0 and sync_rows > 0:
                diff = n_frames - sync_rows
                if diff == 0:
                    print('  OK PERFECT MATCH: Video frames == PPG sync rows')
                else:
                    print(f'  WARNING MISMATCH: Difference of {abs(diff)} rows ({diff:+d})')
        except Exception as e:
            print(f'  Error reading PPG sync: {e}')
    else:
        print(f'  PPG sync not found: {ppg_sync_full}')
    
    print()

## 10. Landmarks Analysis with MediaPipe - 3 View Cases with ROI Boxes

In [ ]:
# Initialize MediaPipe and analyze 3 different view cases
print('=' * 80)
print('LANDMARKS ANALYSIS WITH MEDIAPIPE - 3 VIEW CASES WITH ROI BOXES')
print('=' * 80)

try:
    import mediapipe as mp
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision
    
    print('OK MediaPipe available!')
    
    # Define ROI landmarks
    rois = {
        'full_face': list(range(468)),
        'forehead': [103, 104, 105, 332, 333, 334, 6, 7, 8, 9, 10],
        'left_eye': list(range(22, 53)),
        'right_eye': list(range(252, 283)),
        'nose': list(range(1, 21)) + list(range(195, 221)),
        'mouth': list(range(60, 81)) + list(range(290, 321)),
        'chin': list(range(150, 171)) + list(range(370, 391)),
        'left_iris': list(range(468, 473)),
        'right_iris': list(range(473, 478))
    }
    
    print('ROI Definitions:')
    for roi_name, landmarks in rois.items():
        print(f'  {roi_name}: {len(landmarks)} landmarks')
    
    # Get 3 different view cases
    view_cases = meta_df['view_type'].unique()[:3]
    print(f'Analyzing {len(view_cases)} different view cases: {view_cases}')
    
    # Initialize MediaPipe
    base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        output_face_blendshapes=True,
        output_facial_transformation_matrixes=True,
        num_faces=1)
    detector = vision.FaceLandmarker.create_from_options(options)
    
    # Process each view case
    for view_case in view_cases:
        print(f'VIEW CASE: {view_case}')
        
        sample = meta_df[meta_df['view_type'] == view_case].dropna(subset=['video_full']).head(1)
        
        if len(sample) > 0:
            row = sample.iloc[0]
            video_full = row['video_full']
            
            if os.path.exists(video_full):
                try:
                    reader = imageio.get_reader(video_full, 'ffmpeg')
                    first_frame = reader.get_next_data()
                    reader.close()
                    
                    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=first_frame)
                    detection_result = detector.detect(mp_image)
                    
                    if detection_result.face_landmarks:
                        landmarks = detection_result.face_landmarks[0]
                        print(f'Detected {len(landmarks)} landmarks')
                        
                        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
                        
                        # Full landmarks
                        ax1.imshow(first_frame)
                        ax1.scatter([p.x * first_frame.shape[1] for p in landmarks],
                                  [p.y * first_frame.shape[0] for p in landmarks],
                                  s=1, c='red', alpha=0.5)
                        ax1.set_title(f'{view_case} - All Landmarks')
                        ax1.axis('off')
                        
                        # ROI boxes (24x24)
                        ax2.imshow(first_frame)
                        for roi_name, roi_landmarks in rois.items():
                            if roi_landmarks and roi_name != 'full_face':
                                valid_indices = [i for i in roi_landmarks if i < len(landmarks)]
                                if valid_indices:
                                    roi_points = [landmarks[i] for i in valid_indices]
                                    x_coords = [p.x * first_frame.shape[1] for p in roi_points]
                                    y_coords = [p.y * first_frame.shape[0] for p in roi_points]
                                    center_x = int(np.mean(x_coords))
                                    center_y = int(np.mean(y_coords))
                                    
                                    # Draw 24x24 box
                                    rect = patches.Rectangle(
                                        (center_x - 12, center_y - 12), 24, 24,
                                        linewidth=2, edgecolor='cyan', facecolor='none')
                                    ax2.add_patch(rect)
                                    ax2.text(center_x - 12, center_y - 15, roi_name,
                                           color='cyan', fontsize=8,
                                           bbox=dict(facecolor='cyan', alpha=0.5))
                        
                        ax2.set_title(f'{view_case} - ROI Boxes (24x24)')
                        ax2.axis('off')
                        plt.suptitle(f'View: {view_case}')
                        plt.tight_layout()
                        plt.show()
                    else:
                        print('No face detected')
                except Exception as e:
                    print(f'Error: {e}')
            else:
                print(f'Video not found: {video_full}')
        else:
            print(f'No sample for view: {view_case}')
    
    print('Landmark analysis complete!')
    
except ImportError as e:
    print(f'MediaPipe not available: {e}')
    print('Install with: pip install mediapipe')

## 11. Multi-Camera Analysis

In [ ]:
print('=' * 80)
print('MULTI-CAMERA ANALYSIS')
print('=' * 80)

if 'camera_type' in meta_df.columns:
    print(f'Camera types: {meta_df["camera_type"].unique()}')
    print(f'Camera distribution:')
    print(meta_df['camera_type'].value_counts())
    
    # Vital signs by camera
    camera_vitals = meta_df.groupby('camera_type')[vital_signs].agg(['mean', 'std', 'min', 'max']).round(2)
    display(camera_vitals)
    
    # Plot vital signs by camera
    for vital in vital_signs[:6]:
        if vital in meta_df.columns:
            plt.figure(figsize=(10, 6))
            sns.boxplot(data=meta_df, x='camera_type', y=vital, palette='Set2')
            plt.title(f'{vital.replace("_", " ").title()} by Camera Type')
            plt.ylabel(vital.replace('_', ' ').title())
            plt.xticks(rotation=45)
            plt.show()

## 12. Condition Analysis (Rest vs Exercise)

In [ ]:
print('=' * 80)
print('CONDITION ANALYSIS (REST vs EXERCISE)')
print('=' * 80)

if 'condition' in meta_df.columns:
    print(f'Conditions: {meta_df["condition"].unique()}')
    print(f'Condition distribution:')
    print(meta_df['condition'].value_counts())
    
    # Vital signs by condition
    condition_vitals = meta_df.groupby('condition')[vital_signs].agg(['mean', 'std', 'min', 'max']).round(2)
    display(condition_vitals)
    
    # Plot vital signs by condition
    for vital in vital_signs[:8]:
        if vital in meta_df.columns:
            plt.figure(figsize=(10, 6))
            sns.boxplot(data=meta_df, x='condition', y=vital, palette='Set2')
            plt.title(f'{vital.replace("_", " ").title()} by Condition')
            plt.ylabel(vital.replace('_', ' ').title())
            plt.show()
    
    # Statistical comparison
    print('Statistical Comparison:')
    conditions = meta_df['condition'].unique()
    for vital in vital_signs:
        if vital in meta_df.columns:
            for cond in conditions:
                data = meta_df[meta_df['condition'] == cond][vital].dropna()
                if len(data) > 0:
                    print(f'  {vital:20s} {cond:8s}: Mean={data.mean():8.2f}')
            if len(conditions) >= 2:
                means = [meta_df[meta_df['condition'] == c][vital].mean() for c in conditions]
                diff = means[1] - means[0] if len(means) > 1 else 0
                print(f'  {vital:20s} DIFF: {diff:+8.2f}')
else:
    print('Condition column not found')

## 13. Summary and Key Findings

In [ ]:
print('=' * 80)
print('SUMMARY AND KEY FINDINGS')
print('=' * 80)

print('DATABASE STRUCTURE:')
print(f'  Total entries: {len(meta_df)}')
print(f'  Total columns: {len(meta_df.columns)}')
print(f'  Unique subjects: {meta_df["patient_id"].nunique()}')
print(f'  Conditions: {meta_df["condition"].unique()}')
print(f'  Camera types: {meta_df["camera_type"].unique()}')
print(f'  View types: {meta_df["view_type"].unique()}')

print()
print('KEY FINDINGS:')
print('  OK ALL METADATA IS IN db.csv (vital signs, demographics, session info)')
print('  OK File paths in db.csv are RELATIVE to DATASET_PATH')
print('  OK Full paths: os.path.join(DATASET_PATH, relative_path_from_db)')
print('  OK PPG files can be .npy or .PW (text) format')
print('  OK Video frame count: use reader.count_frames()')
print('  OK MediaPipe works for landmark detection')
print('  OK ROI boxes (24x24) plotted for each view case')

print()
print('FILE PATH EXAMPLES:')
print('  db.csv["ecg"]:     ecg/1020_after.json')
print('  db.csv["ppg"]:     ppg/1020_after.PW')
print('  db.csv["video"]:   video/1020_FullHDwebcam_after.avi')
print('  db.csv["meta"]:    meta/1020_FullHDwebcam_after.txt')
print('  db.csv["ppg_sync"]: ppg_sync/1020_FullHDwebcam_after.txt')
print()
print('  Full path: DATASET_PATH + relative_path')
print(f'  = {DATASET_PATH} + ecg/1020_after.json')

print()
print('NEXT STEPS:')
print('  1. Use meta_df with full paths for all file access')
print('  2. Process videos with imageio[ffmpeg]')
print('  3. Load PPG signals with np.load() or np.loadtxt()')
print('  4. Extract landmarks with MediaPipe')
print('  5. Align signals using timestamps from meta/ppg_sync files')